# The Geometry of Superposition

*Book two of the superposition series. Book one
([`toy_models_of_superposition.ipynb`](./toy_models_of_superposition.ipynb)) showed that
sparsity turns superposition on. This notebook asks the question book one left open:
when a model superposes features, **what shape do they make — and why?***

Reference: Elhage et al. 2022, [*Toy Models of
Superposition*](https://transformer-circuits.pub/2022/toy_model/index.html) — the
"Geometry of Superposition" and "Learning Dynamics" sections. Hyperparameters are pinned
from the paper's public Colab.

**Hypothesis.** Superposition is not amorphous. Features arrange into *uniform
polytopes* — digons, triangles, tetrahedra, pentagons, square antiprisms — measurable as
fractional *feature dimensionality* clinging to ½, ⅔, ¾, ⅖, ⅜. Training reaches these
configurations through discrete *energy-level jumps*, and non-uniformity deforms the
geometry smoothly until it snaps.

*Scope note: like book one, this notebook is pure synthetic toy — no real language
model, and no Arabic; the dialect thread resumes once these tools meet real models.*

## Act 0 — The question book one left open

Book one ended on a pentagon: five features, two hidden dimensions, high sparsity — and
the trained weight columns landed at five *equal* angles, 72° apart. We treated that as
an observation. But nothing in the loss says "be regular." Why not four features crammed
and one straggler? Why the most symmetric arrangement available?

Here is the tease: our model is solving a physics problem. Place charged particles on a
sphere and let them repel — they settle into maximally-symmetric configurations (the
[Thomson problem](https://en.wikipedia.org/wiki/Thomson_problem)). Interference between
features acts like repulsion between charges. By the end of this notebook that analogy
will be quantitative: we will *measure* the fraction of a dimension each feature gets,
watch those fractions cling to a handful of exact values (½, ⅔, ¾, ⅖, ⅜), and identify
the polytope behind each value.

The plan: rebuild the toy (Act 1) → build the measuring instrument and calibrate it on
known solutions (Act 2) → sweep sparsity and find the plateaus (Act 3) → show the
plateaus are polytopes (Act 4) → watch training jump between them (Act 5) → perturb one
feature and watch the geometry stretch and snap (Act 6).

In [ ]:
# lib: imports
from pathlib import Path

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

SEED = 0
torch.manual_seed(SEED)
CACHE = Path("cache")  # created by the cells that write to it (keeps pytest's cwd clean)
print("torch", torch.__version__, "| seed", SEED)